In [ ]:
import os
import sys
import subprocess
import random
import json
import requests

# 1. Mount Drive
from google.colab import drive, userdata
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# 2. Setup Repo (Idempotent)
REPO_DIR = "/content/teachmewrite"
REPO_URL = "https://github.com/edmedpublic-hub/teachmewrite.git"

if os.path.exists(REPO_DIR):
    print("Repo exists. Pulling latest changes...")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", "main"], capture_output=True)
else:
    print("Cloning repo...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

# 3. Install dependencies (ANTHROPIC INCLUDED THIS TIME)
print("Installing dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "requests", "anthropic"], check=True)

# 4. Load Secrets
try:
    os.environ["GITHUB_TOKEN"] = userdata.get('GITHUB_TOKEN')
    os.environ["ANTHROPIC_API_KEY"] = userdata.get('ANTHROPIC_API_KEY')
    os.environ["GCP_PROJECT"] = userdata.get('GCP_PROJECT')
    print("Secrets loaded successfully.")
except Exception as e:
    print(f"ERROR loading secrets: {e}")
    raise

# 5. Set seeds
random.seed(42)

# 6. Define Live URL
LIVE_URL = "https://teachmewrite-ai-195673983724.asia-south1.run.app"

# 7. Assertions
assert os.path.exists(f"{REPO_DIR}/ai-service/main.py"), "main.py not found!"
assert os.path.exists(f"{REPO_DIR}/ai-service/golden_set.jsonl"), "golden_set.jsonl not found!"
assert os.environ.get("ANTHROPIC_API_KEY") is not None, "ANTHROPIC_API_KEY is missing!"
print(f"Target Live Service: {LIVE_URL}")
print("✅ Cell 1a complete. Environment is ready.")

Mounted at /content/drive
Cloning repo...
Installing dependencies...
Secrets loaded successfully.
Target Live Service: https://teachmewrite-ai-195673983724.asia-south1.run.app
✅ Cell 1a complete. Environment is ready.


In [ ]:
from anthropic import Anthropic

# Initialize the Judge client
judge_client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

# Define the Judge Prompt (with escaped braces for .format())
JUDGE_PROMPT = """You are a Senior English Teacher and AI Evaluator for the Pakistani education system.
You will be given a STUDENT TEXT and the AI FEEDBACK generated by an automated system.

Your job is to grade the AI FEEDBACK on three criteria using a scale of 1 to 5:
1. ACCURACY: Did the AI correctly identify the actual grammar/clarity errors in the student's text?
2. HELPFULNESS: Is the explanation actionable for a Pakistani student learning English?
3. TONE: Is the feedback professional, respectful, and encouraging?

Return ONLY a JSON object with these keys (no markdown, no extra text):
{{
  "accuracy_score": int,
  "helpfulness_score": int,
  "tone_score": int,
  "reasoning": "A brief 1-2 sentence explanation of your scores."
}}

STUDENT TEXT:
{student_text}

AI FEEDBACK:
{ai_feedback}
"""

def evaluate_batch(golden_set_path, live_url, limit=None):
    """
    Evaluate all cases in the golden set against the live service and the Judge.
    """
    results = []

    with open(golden_set_path, 'r') as f:
        cases = [json.loads(line) for line in f if line.strip()]

    if limit:
        cases = cases[:limit]

    print(f"Evaluating {len(cases)} cases...\n")

    for i, case in enumerate(cases, 1):
        case_id = case.get("id", f"case_{i}")
        student_text = case["student_text"]
        level = case.get("level", 5)

        # 1. Call the live service
        try:
            api_resp = requests.post(
                f"{live_url}/feedback",
                json={"student_text": student_text, "level": level},
                timeout=45
            )
            ai_feedback = api_resp.json()
        except Exception as e:
            print(f"[{i:02d}/{len(cases)}] {case_id}  API ERROR: {e}")
            continue

        # 2. Call the Judge
        try:
            formatted = JUDGE_PROMPT.format(
                student_text=student_text,
                ai_feedback=json.dumps(ai_feedback, indent=2)
            )
            judge_resp = judge_client.messages.create(
                model="claude-haiku-4-5-20251001",
                max_tokens=500,
                system="You are a strict English teacher evaluator. Respond only with valid JSON.",
                messages=[{"role": "user", "content": formatted}]
            )

            # Defensive parsing
            raw = judge_resp.content[0].text.strip()
            raw = raw.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
            judge_eval = json.loads(raw)

            # Validate scores
            for k in ["accuracy_score", "helpfulness_score", "tone_score"]:
                assert 1 <= judge_eval[k] <= 5, f"{k} out of bounds: {judge_eval[k]}"

            results.append({
                "case_id": case_id,
                "student_text": student_text,
                "ai_feedback": ai_feedback,
                "judge_eval": judge_eval
            })

            print(f"[{i:02d}/{len(cases)}] {case_id}  Acc:{judge_eval['accuracy_score']} Help:{judge_eval['helpfulness_score']} Tone:{judge_eval['tone_score']}")

        except Exception as e:
            print(f"[{i:02d}/{len(cases)}] {case_id}  JUDGE ERROR: {e}")
            continue

    return results

print("✅ Cell 1b complete. evaluate_batch() is ready.")

✅ Cell 1b complete. evaluate_batch() is ready.


In [ ]:
GOLDEN_SET_PATH = f"{REPO_DIR}/ai-service/golden_set.jsonl"

# Smoke test: first 3 cases only
smoke_results = evaluate_batch(GOLDEN_SET_PATH, LIVE_URL, limit=3)

assert len(smoke_results) >= 2, "Smoke test failed: too few successful results"
print("\n✅ Cell 1c complete. Smoke test passed.")

Evaluating 3 cases...

[01/3] g01  Acc:5 Help:5 Tone:5
[02/3] g02  Acc:5 Help:5 Tone:5
[03/3] g03  Acc:5 Help:4 Tone:5

✅ Cell 1c complete. Smoke test passed.


In [ ]:
# Full batch evaluation on all 20 cases
full_results = evaluate_batch(GOLDEN_SET_PATH, LIVE_URL)

# Summary statistics
assert len(full_results) >= 18, f"Too many failures: only {len(full_results)}/20 succeeded"

acc_scores = [r["judge_eval"]["accuracy_score"] for r in full_results]
help_scores = [r["judge_eval"]["helpfulness_score"] for r in full_results]
tone_scores = [r["judge_eval"]["tone_score"] for r in full_results]

print("\n" + "="*50)
print("BATCH EVALUATION SUMMARY")
print("="*50)
print(f"Cases evaluated: {len(full_results)}/20")
print(f"Accuracy:    avg {sum(acc_scores)/len(acc_scores):.2f}  |  min {min(acc_scores)}  |  max {max(acc_scores)}")
print(f"Helpfulness: avg {sum(help_scores)/len(help_scores):.2f}  |  min {min(help_scores)}  |  max {max(help_scores)}")
print(f"Tone:        avg {sum(tone_scores)/len(tone_scores):.2f}  |  min {min(tone_scores)}  |  max {max(tone_scores)}")

# Identify weak cases (any score below 4)
weak_cases = [
    r for r in full_results
    if any(r["judge_eval"][k] < 4 for k in ["accuracy_score", "helpfulness_score", "tone_score"])
]

if weak_cases:
    print("\nWeak cases (any score < 4):")
    for w in weak_cases:
        ev = w["judge_eval"]
        print(f"  - {w['case_id']}: Acc:{ev['accuracy_score']} Help:{ev['helpfulness_score']} Tone:{ev['tone_score']}")
        print(f"    Reasoning: {ev['reasoning']}")
else:
    print("\n✅ No weak cases. All scores >= 4 across all cases.")

assert len(acc_scores) > 0, "No accuracy scores collected"
print("\n✅ Cell 1d complete. Full batch evaluated.")

Evaluating 20 cases...

[01/20] g01  Acc:5 Help:5 Tone:5
[02/20] g02  Acc:5 Help:5 Tone:5
[03/20] g03  Acc:5 Help:5 Tone:4
[04/20] g04  Acc:5 Help:5 Tone:5
[05/20] g05  Acc:5 Help:4 Tone:5
[06/20] g06  Acc:5 Help:5 Tone:5
[07/20] g07  Acc:5 Help:5 Tone:4
[08/20] g08  Acc:4 Help:5 Tone:5
[09/20] g09  Acc:4 Help:4 Tone:5
[10/20] g10  Acc:5 Help:5 Tone:5
[11/20] g11  Acc:5 Help:5 Tone:5
[12/20] g12  Acc:5 Help:5 Tone:5
[13/20] g13  Acc:5 Help:5 Tone:5
[14/20] g14  Acc:5 Help:5 Tone:5
[15/20] g15  Acc:5 Help:4 Tone:5
[16/20] g16  Acc:5 Help:4 Tone:5
[17/20] g17  Acc:5 Help:4 Tone:5
[18/20] g18  Acc:5 Help:4 Tone:5
[19/20] g19  Acc:5 Help:5 Tone:5
[20/20] g20  Acc:5 Help:4 Tone:5

BATCH EVALUATION SUMMARY
Cases evaluated: 20/20
Accuracy:    avg 4.90  |  min 4  |  max 5
Helpfulness: avg 4.65  |  min 4  |  max 5
Tone:        avg 4.90  |  min 4  |  max 5

✅ No weak cases. All scores >= 4 across all cases.

✅ Cell 1d complete. Full batch evaluated.


In [ ]:
artifact = {
    "run_date": "2026-10-21",
    "model_under_test": "claude-haiku-4-5 (production service)",
    "judge_model": "claude-haiku-4-5-20251001",
    "num_cases": len(full_results),
    "summary": {
        "accuracy_avg": sum(acc_scores)/len(acc_scores),
        "helpfulness_avg": sum(help_scores)/len(help_scores),
        "tone_avg": sum(tone_scores)/len(tone_scores),
        "weak_cases": 0
    },
    "detailed_results": full_results
}

save_path = "/content/drive/MyDrive/teachmewrite-ai/batch_eval_day88.json"
with open(save_path, "w") as f:
    json.dump(artifact, f, indent=2)

print(f"Saved batch evaluation to {save_path}")
assert os.path.exists(save_path), "Save failed!"
print("✅ Day 88 artifact saved.")

Saved batch evaluation to /content/drive/MyDrive/teachmewrite-ai/batch_eval_day88.json
✅ Day 88 artifact saved.
